# Sequence Forecasting with RNN, TimeDistributed training and Attention — STUDENT SKETCH

This notebook has two deliberately connected parts:

1. **LIVE** — Chicago CTA ridership, closely aligned with Aurélien Géron's Chapter 13 notebook. We add a short stationarity analysis and reproduce the chapter's sequence-to-sequence / **TimeDistributed** training idea (`56 -> 14`).
2. **HOMEWORK (groups ≤ 3)** — multi-station daily weather forecasting with Meteostat. The model sees **60 days** and forecasts **21 future days** for temperature, precipitation and wind. Training is TimeDistributed: every recurrent timestep produces its own 21-day forecast.

Reference for the live activity: https://colab.research.google.com/github/serivan/DeepLearning/blob/master/PyTorch/13_processing_sequences_using_rnns_and_cnns.ipynb

**Important terminology.** In PyTorch there is no need for a Keras-style `TimeDistributed` wrapper around `nn.Linear`: applying `nn.Linear` to `[B,T,H]` automatically applies the same linear map to the last dimension at every timestep.


## How to use this notebook

- The **complete** version contains all implementations and explanations.
- The **student sketch** has data utilities, preprocessing, plotting and training helpers already completed. TODOs are concentrated on recurrence, TimeDistributed outputs and attention.
- The live section is intentionally close to Géron's notebook so the book/notebook can be used as a reference during class.
- The homework dataset is supplied as frozen CSV files; students are not expected to query Meteostat.


## Learning objectives

By the end, students should be able to:

- distinguish trend/seasonality from weak stationarity and interpret ADF/KPSS cautiously;
- reason about tensors shaped `[batch, time, features]`;
- explain the recurrent hidden state and the outputs returned by `nn.RNN` / `nn.LSTM`;
- build sequence-to-sequence labels with sliding future horizons;
- understand **TimeDistributed training**: `[B,T,H] -> [B,T,horizon,targets]`;
- distinguish training supervision at every timestep from operational inference using the final timestep;
- implement causal temporal attention without future leakage;
- evaluate 21-day forecasts by target, horizon and unseen station.


# Part I — LIVE: Géron + stationarity + TimeDistributed training

The live section uses the same CTA ridership data and the same key choices as Géron's sequence-forecasting progression: the `2016–2018` training period, `2019-01` to `2019-05` validation period, **56-day input window**, multivariate inputs (`rail`, `bus`, next-day type) and a **14-day** forecasting horizon.


### LIVE ↔ Géron reference map

| This notebook | Géron Chapter 13 |
|---|---|
| CTA download/cleaning + provided visual EDA | *Forecasting a Time Series* |
| weekly baseline | naive seasonal forecasting |
| 56-day windows | `TimeSeriesDataset` |
| multivariate input | `rail`, `bus`, `next_day_type` |
| manual recurrence / `nn.RNN` | simple RNN section |
| `[B,56,14]` labels | `Seq2SeqDataset` + `unfold()` |
| Linear on `[B,56,H]` | `Seq2SeqRnnModel`: same output layer at every timestep |
| attention | **extension added here**, not the core Chapter 13 sequence |


In [ ]:
# LIVE setup
from pathlib import Path
import math, random, copy, time, tarfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

In [ ]:
# [PROVIDED] Download and clean the CTA ridership data used by Géron.
GERON_DIR = Path("datasets/ridership")
GERON_CSV = GERON_DIR / "CTA_-_Ridership_-_Daily_Boarding_Totals.csv"


def load_geron_ridership():
    if not GERON_CSV.exists():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        tgz = Path("datasets/ridership.tgz")
        if not tgz.exists():
            urllib.request.urlretrieve(
                "https://github.com/ageron/data/raw/main/ridership.tgz", tgz
            )
        with tarfile.open(tgz) as archive:
            # In recent Python versions filter='data' avoids unsafe extraction paths.
            try:
                archive.extractall(path="datasets", filter="data")
            except TypeError:  # compatibility with older Python
                archive.extractall(path="datasets")

    df = pd.read_csv(GERON_CSV, parse_dates=["service_date"])
    df.columns = ["date", "day_type", "bus", "rail", "total"]
    df = (df.sort_values("date")
            .set_index("date")
            .drop(columns="total")
            .drop_duplicates())
    return df

ridership = load_geron_ridership()
ridership.head()

## L0. Visual exploration of the CTA data **[PROVIDED]**

These plots are intentionally **already completed**. During the live session, students should focus on reading the data rather than writing visualization code.

Look for:

- long-term changes in bus and rail ridership;
- weekly periodicity and differences between weekdays/weekends;
- the meaning of the `day_type` variable used later as an input feature;
- the relationship between bus and rail ridership;
- the distinction between short-term variability and smoother trends;
- what a **56-day input window** and a **14-day forecasting horizon** look like on the actual series.


In [ ]:
# [PROVIDED] Data used in the LIVE visual exploration.
live_df = ridership.loc["2016-01":"2019-05"].copy()
live_df["weekday"] = live_df.index.day_name()

WEEKDAY_ORDER = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

print(f"LIVE period: {live_df.index.min().date()} -> {live_df.index.max().date()}")
print(f"Number of daily observations: {len(live_df):,}")
print("Day-type codes:", sorted(live_df["day_type"].dropna().unique()))
live_df.head()


In [ ]:
# [PROVIDED] Plot 1 — Full modelling period: bus and rail ridership.
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(live_df.index, live_df["rail"], label="Rail", linewidth=1.0)
ax.plot(live_df.index, live_df["bus"], label="Bus", linewidth=1.0, alpha=0.8)
ax.set(
    title="CTA ridership over the LIVE modelling period",
    xlabel="Date",
    ylabel="Daily boardings",
)
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# [PROVIDED] Plot 2 — Zoom: daily structure is easier to see over a short interval.
zoom = live_df.loc["2019-03":"2019-05"]
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(zoom.index, zoom["rail"], marker=".", markersize=3, label="Rail")
ax.plot(zoom.index, zoom["bus"], marker=".", markersize=3, label="Bus", alpha=0.8)
ax.set(
    title="Zoom on daily ridership: March–May 2019",
    xlabel="Date",
    ylabel="Daily boardings",
)
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# [PROVIDED] Plot 3 — Average weekly profile.
weekday_means = (
    live_df.groupby("weekday")[["rail", "bus"]]
    .mean()
    .reindex(WEEKDAY_ORDER)
)

ax = weekday_means.plot(kind="bar", figsize=(11, 4))
ax.set(
    title="Average ridership by day of week",
    xlabel="Day of week",
    ylabel="Mean daily boardings",
)
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# [PROVIDED] Plot 4 — Distribution of rail ridership by CTA day_type.
day_types = sorted(live_df["day_type"].dropna().unique())
values = [live_df.loc[live_df["day_type"] == code, "rail"].dropna().values for code in day_types]

fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot(values, labels=day_types, showfliers=False)
ax.set(
    title="Rail ridership distribution by CTA day_type",
    xlabel="day_type code",
    ylabel="Daily rail boardings",
)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

print("Frequency of day_type codes:")
display(live_df["day_type"].value_counts().sort_index().rename("days").to_frame())


In [ ]:
# [PROVIDED] Plot 5 — Bus and rail ridership move together, but not perfectly.
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(live_df["bus"], live_df["rail"], s=10, alpha=0.35)
ax.set(
    title="Bus vs rail ridership",
    xlabel="Bus boardings",
    ylabel="Rail boardings",
)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print("Pearson correlation (bus, rail):", round(live_df[["bus", "rail"]].corr().iloc[0, 1], 3))


In [ ]:
# [PROVIDED] Plot 6 — Raw series and rolling means reveal different time scales.
rail = live_df["rail"]
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(rail.index, rail, label="Daily rail", alpha=0.30, linewidth=0.8)
ax.plot(rail.index, rail.rolling(7).mean(), label="7-day mean", linewidth=1.3)
ax.plot(rail.index, rail.rolling(28).mean(), label="28-day mean", linewidth=1.6)
ax.set(
    title="Rail ridership at different temporal resolutions",
    xlabel="Date",
    ylabel="Daily boardings",
)
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# [PROVIDED] Plot 7 — Visualizing the later supervised-learning setup: 56 input days -> 14 future days.
WINDOW = 56
HORIZON = 14
example_end = pd.Timestamp("2019-03-31")
example_start = example_end - pd.Timedelta(days=WINDOW - 1)
target_start = example_end + pd.Timedelta(days=1)
target_end = example_end + pd.Timedelta(days=HORIZON)

segment = ridership.loc[example_start:target_end, "rail"]

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(segment.index, segment.values, linewidth=1.3)
ax.axvspan(example_start, example_end, alpha=0.18, label="56-day input window")
ax.axvspan(target_start, target_end, alpha=0.28, label="14-day target horizon")
ax.axvline(example_end, linestyle="--", linewidth=1)
ax.set(
    title="From time series to supervised forecasting example",
    xlabel="Date",
    ylabel="Rail boardings",
)
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print("Input :", example_start.date(), "->", example_end.date())
print("Target:", target_start.date(), "->", target_end.date())


### Visual interpretation prompts

No code is required here. Before moving to stationarity and modelling, discuss:

- Which periodic pattern is most obvious in the short-term plot?
- Are bus and rail equally variable?
- Which days systematically have lower ridership?
- Why might `day_type` help an RNN predict future ridership?
- What information is lost when the daily series is replaced by a 28-day rolling mean?
- Looking at the `56 -> 14` plot, which part is input and which part becomes the supervised target?


## L1. Stationarity analysis (LIVE only)

A weakly stationary series has approximately constant mean, variance and autocovariance structure over time. We use three complementary diagnostics:

1. **rolling mean/std** — visual check for changes in level/variance;
2. **ACF** — exposes persistence and periodicity such as the 7-day cycle;
3. **ADF + KPSS** — complementary hypothesis tests.

Interpretation:

- **ADF** null hypothesis: unit root / non-stationarity. Small p-value → evidence *against* the unit-root null.
- **KPSS** null hypothesis: level/trend stationarity (depending on `regression`). Small p-value → evidence *against* stationarity.

Do not treat either test as an oracle: strong seasonality, structural breaks (e.g. COVID) and large samples can make simple binary conclusions misleading.


In [ ]:
# [PROVIDED] Stationarity diagnostics.
# statsmodels is available in standard Colab runtimes; if needed: %pip install -q statsmodels
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf


def stationarity_report(series, rolling=28, acf_lags=56, title="series"):
    s = pd.Series(series).dropna().astype(float)

    fig, axes = plt.subplots(2, 1, figsize=(12, 7))
    axes[0].plot(s.index, s.values, alpha=.55, label="series")
    axes[0].plot(s.rolling(rolling).mean(), label=f"{rolling}d rolling mean")
    axes[0].plot(s.rolling(rolling).std(), label=f"{rolling}d rolling std")
    axes[0].set_title(title)
    axes[0].legend()
    plot_acf(s, lags=acf_lags, ax=axes[1])
    axes[1].set_title("Autocorrelation")
    plt.tight_layout(); plt.show()

    adf_stat, adf_p, *_ = adfuller(s, autolag="AIC")
    kpss_stat, kpss_p, *_ = kpss(s, regression="c", nlags="auto")
    return pd.DataFrame({
        "test": ["ADF (H0: unit root)", "KPSS (H0: stationary)"],
        "statistic": [adf_stat, kpss_stat],
        "p_value": [adf_p, kpss_p],
    })

live_stationarity = stationarity_report(
    ridership.loc["2016-01":"2019-05", "rail"],
    rolling=28, acf_lags=56, title="CTA rail ridership"
)
live_stationarity

### Interpretation prompt

Discuss before modelling:

- Is the raw rail series well described as stationary?
- What does the ACF reveal around lag 7, 14, 21, …?
- Why can a weekly seasonal baseline be strong even if the raw series is not stationary?
- Which structural break later in the full CTA dataset would make stationarity assumptions especially fragile?


## L2. Weekly baseline and one-step windows

The weekly baseline is

\[
\hat y_t = y_{t-7}.
\]

It is a useful minimum benchmark because CTA usage has a strong weekly rhythm.


In [ ]:
# [PROVIDED] Weekly naive validation error.
rail_valid_pd = ridership.loc["2019-01":"2019-05", "rail"]
weekly_pred = rail_valid_pd.shift(7)
mask = weekly_pred.notna()
weekly_mae = (rail_valid_pd[mask] - weekly_pred[mask]).abs().mean()
print(f"Weekly naive MAE: {weekly_mae:,.0f} riders/day")

In [ ]:
# [PROVIDED] One-step dataset analogous to Géron's TimeSeriesDataset.
class TimeSeriesDataset(Dataset):
    def __init__(self, series, window_length):
        self.series = series
        self.window_length = window_length

    def __len__(self):
        return len(self.series) - self.window_length

    def __getitem__(self, idx):
        end = idx + self.window_length
        return self.series[idx:end], self.series[end]

GERON_WINDOW = 56
rail_train = torch.tensor(
    ridership.loc["2016-01":"2018-12", ["rail"]].values / 1e6,
    dtype=torch.float32,
)
rail_valid = torch.tensor(
    ridership.loc["2019-01":"2019-05", ["rail"]].values / 1e6,
    dtype=torch.float32,
)

one_train = TimeSeriesDataset(rail_train, GERON_WINDOW)
one_loader = DataLoader(one_train, batch_size=32, shuffle=True)
X_one, y_one = next(iter(one_loader))
print("one-step X:", X_one.shape, "y:", y_one.shape)

## L3. Manual RNN recurrence

At timestep `t`:

\[
h_t = \tanh(W[x_t;h_{t-1}] + b).
\]

Key Python/PyTorch options:

- `X.shape == [B,T,F]` because we use batch-first data;
- `X.transpose(0, 1)` changes it to `[T,B,F]`, making a Python loop over time straightforward;
- `torch.cat(..., dim=1)` concatenates features, not batches;
- the same `memory_cell` parameters are reused at every timestep.


In [ ]:
class ManualRNNOneStep(nn.Module):
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()
        self.hidden_size = hidden_size
        self.memory_cell = nn.Sequential(
            nn.Linear(input_size + hidden_size, hidden_size),
            nn.Tanh(),
        )
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, X):
        B, T, F = X.shape
        H = torch.zeros(B, self.hidden_size, device=X.device)
        for X_t in X.transpose(0, 1):
            # TODO LIVE 1: concatenate current input X_t and previous state H
            XH = ...
            # TODO LIVE 1: update H using the same recurrent cell
            H = ...
        return self.output(H)

# manual_rnn = ManualRNNOneStep().to(DEVICE)

## L4. Replace the Python loop with `nn.RNN`

With `batch_first=True`, PyTorch receives `[B,T,F]` and returns:

- `sequence_output`: `[B,T,H]`, one hidden representation per timestep;
- `last_state`: `[num_layers,B,H]`.

For one-step forecasting we can use `sequence_output[:, -1]`. The full `[B,T,H]` tensor becomes central in the TimeDistributed section.


In [ ]:
class BuiltinRNNOneStep(nn.Module):
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, X):
        # TODO LIVE 2: call self.rnn(X)
        sequence_output, last_state = ...
        # TODO LIVE 2: use only the final timestep for one-step forecasting
        last = ...
        return self.output(last)

## L5. Multivariate inputs, as in Géron

Géron uses:

- rail ridership;
- bus ridership;
- **tomorrow's day type**, known from the calendar and one-hot encoded.

After scaling ridership by `1e6` and one-hot encoding the three day types, the recurrent input has **5 features**.


In [ ]:
# [PROVIDED] Same multivariate construction used in Géron's notebook.
df_mulvar = ridership[["rail", "bus"]].copy() / 1e6
df_mulvar["next_day_type"] = ridership["day_type"].shift(-1)
df_mulvar = pd.get_dummies(df_mulvar, dtype=float).dropna()

mulvar_train = torch.tensor(df_mulvar.loc["2016-01":"2018-12"].values, dtype=torch.float32)
mulvar_valid = torch.tensor(df_mulvar.loc["2019-01":"2019-05"].values, dtype=torch.float32)
print("features:", list(df_mulvar.columns))
print("train tensor:", mulvar_train.shape)

## L6. Géron-style TimeDistributed training: `56 -> 14`

This is the core sequence-to-sequence construction.

For each 56-day input window, the target is **not** only one 14-day vector. Instead, each input timestep has its own next-14-day target:

\[
X \in \mathbb{R}^{56\times5},\qquad
Y \in \mathbb{R}^{56\times14}.
\]

At input timestep 0, the RNN predicts days `+1…+14` relative to that timestep; at timestep 55 it predicts the 14 days after the end of the full window.

`Tensor.unfold(dimension=0, size=14, step=1)` creates these overlapping future windows efficiently.


In [ ]:
class GeronSeq2SeqDataset(Dataset):
    def __init__(self, series, window_length=56, horizon=14):
        self.series = series
        self.window_length = window_length
        self.horizon = horizon

    def __len__(self):
        return len(self.series) - self.window_length - self.horizon + 1

    def __getitem__(self, idx):
        end = idx + self.window_length
        window = self.series[idx:end]
        target_period = self.series[idx + 1:end + self.horizon, 0]
        # TODO LIVE 3: create all overlapping 14-day targets with unfold()
        target = ...
        return window, target

# seq_train_ds = GeronSeq2SeqDataset(mulvar_train, 56, 14)

### Why this is “TimeDistributed” in PyTorch

`nn.RNN` returns `[B,56,H]`. If `self.output = nn.Linear(H,14)`, then:

```text
[B,56,H] --Linear(H,14)--> [B,56,14]
```

The **same** linear layer is applied at every timestep because `nn.Linear` always transforms the last dimension. No extra wrapper is required.


In [ ]:
class TimeDistributedRNN(nn.Module):
    def __init__(self, input_size=5, hidden_size=32, horizon=14):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, horizon)

    def forward(self, X):
        # TODO LIVE 4: get every recurrent output [B,T,H]
        outputs, last_state = ...
        # TODO LIVE 4: apply the same Linear layer to every timestep
        return ...

# During training: criterion(model(X), y) uses ALL 56 timesteps.
# During operational inference: forecast = model(X)[:, -1, :]

## L7. Attention extension

The TimeDistributed RNN gives dense supervision, but its final operational forecast still uses the final recurrent representation. As an extension, we can aggregate all recurrent outputs using attention.

For the live activity we use **final-origin attention** (one attention vector over the 56-day input) to predict the final 14-day horizon. The homework later introduces **causal TimeDistributed attention**, where every timestep gets a context that can only attend to its own past.


In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, hidden_size, attention_size=32):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(hidden_size, attention_size), nn.Tanh(), nn.Linear(attention_size, 1)
        )

    def forward(self, sequence_output):
        # TODO LIVE 5: scalar score per timestep -> [B,T]
        scores = ...
        # TODO LIVE 5: normalize across time (which dimension?)
        weights = ...
        # TODO LIVE 5: weighted sum of hidden states -> [B,H]
        context = ...
        return context, weights

### Live take-away

The central progression is now:

```text
one-step windows
    ↓
manual recurrent state
    ↓
nn.RNN outputs [B,T,H]
    ↓
Géron Seq2Seq labels [B,T,14]
    ↓
Linear(H,14) applied at every timestep = TimeDistributed training
    ↓
final forecast = predictions[:, -1]
    ↓
attention as an extension
```

This is the exact conceptual bridge to the weather homework.


# Part II — HOMEWORK: multi-station weather, `60 -> 21`, TimeDistributed

**Group size:** maximum 3 students.

### Problem

Use a global model trained on multiple Meteostat stations to forecast the next **21 daily values** of:

1. mean temperature (`temp`, °C),
2. precipitation (`prcp`, mm/day),
3. mean wind speed (`wspd`, km/h).

Default context: **60 days**.

The model is trained TimeDistributed: each of the 60 recurrent outputs predicts its own next 21 days. Therefore:

```text
X:       [B, 60, F]
labels:  [B, 60, 21, 3]
pred:    [B, 60, 21, 3]
```

For operational validation/test, use only:

```python
pred[:, -1]   # [B,21,3]
```

which corresponds to the forecast issued after the final observed day.


## H0. Experimental protocol

- **Training/development period:** observations up to **28 February 2026**.
- **Historical test period:** **1 March 2026 → 18 October 2026**.
- Within the development period, the last `VAL_DAYS` are reserved for validation/model selection.
- Lombardy stations marked `train` are used for training, validation and the temporal historical test.
- Separate Lombardy stations are held out completely as `spatial_test_lombardia`.
- Capitanata (Puglia) stations are held out as `spatial_test_capitanata`, providing a stronger geographic/climatic transfer test.
- Observations after **18 October 2026** are reserved for the later real-future evaluation.
- Students may either use the supplied CSV or reproduce/extend the observations online with Meteostat.

The preprocessing statistics and model weights must be fitted **only on rows whose phase is `train`**.


In [ ]:
# Homework configuration
from pathlib import Path
import json
import pandas as pd
from sklearn.preprocessing import StandardScaler

DATA_START = pd.Timestamp("2015-01-01")
TRAIN_END = pd.Timestamp("2026-02-28")
TEST_START = pd.Timestamp("2026-03-01")
TEST_END = pd.Timestamp("2026-10-18")
FUTURE_START = TEST_END + pd.Timedelta(days=1)

INPUT_DAYS = 60
HORIZON = 21
TARGETS = ["temp", "prcp", "wspd"]
N_TARGETS = len(TARGETS)
RAIN_THRESHOLD_MM = 0.5
VAL_DAYS = 180

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# The instructor can provide the historical data as CSV.
PROVIDED_CSV = DATA_DIR / "weather_multistation_daily.csv"
DOWNLOADED_CSV = DATA_DIR / "weather_multistation_downloaded.csv"

# "csv"      -> use the instructor-supplied file
# "download" -> download the same stations from Meteostat and save them locally
DATA_SOURCE = "csv"
DOWNLOAD_TO_TODAY = True

print("Input:", INPUT_DAYS, "days | horizon:", HORIZON, "days | targets:", TARGETS)
print("Train data end:", TRAIN_END.date())
print("Historical test:", TEST_START.date(), "->", TEST_END.date())
print("Future test starts:", FUTURE_START.date())
print("Data source:", DATA_SOURCE)


## H1. Data acquisition — choose **CSV** or **online download** [PROVIDED]

The assignment can be run in two equivalent ways.

**Option A — supplied CSV (recommended / fully reproducible).**  Use the single file distributed by the instructor. It already contains station metadata (`station_id`, `region_group`, `split`, latitude, longitude and elevation) together with daily observations.

**Option B — Meteostat download.**  Read the official station IDs from the supplied CSV, download the same stations directly from Meteostat, and save the result locally. If the notebook is executed after **18 October 2026**, the downloaded file may contain newer observations. Those rows are kept for future evaluation but are automatically excluded from model fitting and hyperparameter tuning.

```text
Meteostat / supplied CSV
        |
        v
all available observations
        |
        +-- date <= 2026-10-18 --> development data (train/validation/spatial tests)
        |
        +-- date >  2026-10-18 --> future observations only; NEVER fit/tune on these rows
```

The Python Meteostat client is used without an API key. `fill=False` preserves missing records instead of fabricating a complete daily grid; `include_model_data=False` is used for the benchmark download so archived model/forecast data are not mixed into the observed-data benchmark.
**Local copy.** In online mode, the notebook writes the downloaded observations to `data/weather_multistation_downloaded.csv`. You can keep that file, reuse it offline, or download it from Colab to your computer.


In [ ]:
# [PROVIDED] CSV / Meteostat acquisition and local saving.
import pandas as pd
import numpy as np

META_COLS = [
    "station_id", "station_name", "region_group", "split",
    "latitude", "longitude", "elevation"
]


def _read_station_config_from_csv(path=PROVIDED_CSV):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Put the instructor CSV in the data/ folder first. "
            "The same file is used to recover the official station IDs in download mode."
        )
    seed = pd.read_csv(path, dtype={"station_id": str})
    missing = [c for c in META_COLS if c not in seed.columns]
    if missing:
        raise ValueError(f"The supplied CSV is missing station metadata columns: {missing}")
    return seed[META_COLS].drop_duplicates("station_id").reset_index(drop=True)


def _configure_meteostat_for_colab(ms):
    # Colab-safe cache location; harmless in normal Jupyter as well.
    cache_dir = Path("/content/meteostat_cache") if Path("/content").exists() else DATA_DIR / "meteostat_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    if hasattr(ms.config, "cache_directory"):
        ms.config.cache_directory = str(cache_dir)
    if hasattr(ms.config, "stations_db_file"):
        ms.config.stations_db_file = str(cache_dir / "stations.db")
    if hasattr(ms.config, "cache_enable"):
        ms.config.cache_enable = True
    if hasattr(ms.config, "include_model_data"):
        ms.config.include_model_data = False


def _as_api_date(value):
    return pd.Timestamp(value).date()


def download_meteostat_daily(station_config, start=DATA_START, end=None,
                             save_path=DOWNLOADED_CSV):
    """Download official homework stations one at a time and save a local CSV."""
    try:
        import meteostat as ms
    except ImportError as exc:
        raise ImportError("Install Meteostat first with: %pip install -q meteostat") from exc

    _configure_meteostat_for_colab(ms)

    start_ts = pd.Timestamp(start).normalize()
    if end is None:
        today = pd.Timestamp.today().normalize()
        end_ts = today if DOWNLOAD_TO_TODAY else TEST_END
    else:
        end_ts = pd.Timestamp(end).normalize()

    frames = []
    skipped = []
    for row in station_config.itertuples(index=False):
        sid = str(row.station_id)
        try:
            ts = ms.daily(sid, _as_api_date(start_ts), _as_api_date(end_ts))
            frame = ts.fetch(fill=False, location=True, clean=True)
        except Exception as exc:
            print(f"WARNING: station {sid} download failed: {exc}")
            skipped.append(sid)
            continue

        if frame is None or frame.empty:
            print(f"WARNING: no daily records returned for station {sid}")
            skipped.append(sid)
            continue

        frame = frame.reset_index()
        if "time" in frame.columns:
            frame = frame.rename(columns={"time": "date"})
        if "date" not in frame.columns:
            raise KeyError(f"Could not identify date column for station {sid}: {list(frame.columns)}")
        if "station" in frame.columns:
            frame = frame.rename(columns={"station": "station_id"})
        if "station_id" not in frame.columns:
            frame["station_id"] = sid
        frame["station_id"] = frame["station_id"].astype(str)
        frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
        frames.append(frame)

    if not frames:
        raise RuntimeError("Meteostat returned no data for the selected stations.")

    weather = pd.concat(frames, ignore_index=True)
    for col in ["latitude", "longitude", "elevation"]:
        if col in weather.columns:
            weather = weather.drop(columns=col)
    weather = weather.merge(station_config, on="station_id", how="left", validate="many_to_one")
    weather = weather.sort_values(["station_id", "date"]).reset_index(drop=True)

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    weather.to_csv(save_path, index=False)
    print(f"Saved {len(weather):,} rows to {save_path.resolve()}")
    if skipped:
        print("Skipped stations:", skipped)
    return weather


def load_homework_weather(source=DATA_SOURCE):
    if source == "csv":
        if not PROVIDED_CSV.exists():
            raise FileNotFoundError(f"Missing {PROVIDED_CSV}. Upload/copy the instructor CSV into data/.")
        weather = pd.read_csv(PROVIDED_CSV, parse_dates=["date"], dtype={"station_id": str})
    elif source == "download":
        station_config = _read_station_config_from_csv(PROVIDED_CSV)
        weather = download_meteostat_daily(station_config)
    else:
        raise ValueError("DATA_SOURCE must be either 'csv' or 'download'.")

    weather["date"] = pd.to_datetime(weather["date"]).dt.normalize()
    weather["station_id"] = weather["station_id"].astype(str)
    weather = weather.sort_values(["station_id", "date"]).reset_index(drop=True)

    cutoff = TEST_END
    development = weather[weather["date"] <= cutoff].copy()
    future = weather[weather["date"] > cutoff].copy()

    assert development.empty or development["date"].max() <= cutoff
    print(f"All rows:          {len(weather):,}")
    print(f"Historical rows:  {len(development):,}  (<= {cutoff.date()})")
    print(f"Future rows:       {len(future):,}  (>  {cutoff.date()}; evaluation only)")
    if not weather.empty:
        print("Last downloaded/loaded date:", weather["date"].max().date())
    return weather, development, future


# In Colab, install ONLY Meteostat. Do not reinstall pandas/numpy/matplotlib.
# %pip install -q meteostat
#
# Load either the supplied CSV or the online data according to DATA_SOURCE:
if PROVIDED_CSV.exists():
    weather_all, df_raw, future_raw = load_homework_weather()
else:
    print(f"Upload/copy {PROVIDED_CSV} and rerun this cell before the EDA section.")


### Local files and Colab

In download mode the raw download is saved as `data/weather_multistation_downloaded.csv`. This is a normal local file in Jupyter; in Google Colab it lives in the current runtime. If you want a copy on your computer, you can optionally run:

```python
from google.colab import files
files.download("data/weather_multistation_downloaded.csv")
```

The official grading protocol is independent of the acquisition route: **no row after 18/10/2026 may be used to fit weights, preprocessing statistics or hyperparameters**.


## H2. Parts A and B are PROVIDED: EDA, split and preprocessing

Students should focus on modelling rather than acquisition or boilerplate. They should nevertheless understand the leakage rules and tensor construction.

The instructor CSV contains three geographic roles:

- `train`: Lombardy stations used for temporal training/validation;
- `spatial_test_lombardia`: unseen Lombardy stations (same-region spatial generalization);
- `spatial_test_capitanata`: unseen Capitanata stations in Puglia (stronger geographic/climatic transfer test).

**Stationarity analysis is optional in the homework.** The required stationarity activity remains in the LIVE section.


## H2A. Visual exploration of the downloaded/provided weather data **[PROVIDED]**

Before preprocessing or training, inspect the actual multi-station dataset. All plotting code below is **provided**: the goal is to understand the data, not to spend homework time writing visualization boilerplate.

The plots cover:

- geographic distribution of the selected stations;
- temporal coverage by station;
- temperature, precipitation and wind time series for representative stations;
- monthly climatology for the three targets;
- missing-data rates by station/variable;
- differences between Lombardy training stations, unseen Lombardy stations and Capitanata stations.

Use these figures to motivate later comments about transfer across stations and the relative difficulty of the three targets.


In [ ]:
# [PROVIDED] Weather-data visual exploration.
import matplotlib.pyplot as plt


def _station_label(row):
    name = row.get("station_name", "")
    sid = str(row.get("station_id", ""))
    return f"{name} ({sid})" if str(name).strip() else sid


def summarize_weather_data(df):
    required = {"date", "station_id", "temp", "prcp", "wspd"}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Missing required columns for EDA: {missing}")

    print("Rows:", f"{len(df):,}")
    print("Stations:", df["station_id"].nunique())
    print("Date range:", df["date"].min().date(), "->", df["date"].max().date())
    if "split" in df.columns:
        print("Stations by split:")
        display(df[["station_id", "split"]].drop_duplicates()["split"].value_counts())

    summary_cols = [c for c in ["station_id", "station_name", "region_group", "split",
                                "latitude", "longitude", "elevation"] if c in df.columns]
    display(df[summary_cols].drop_duplicates("station_id").sort_values("station_id"))


def plot_station_locations(df):
    meta_cols = [c for c in ["station_id", "station_name", "region_group", "split",
                              "latitude", "longitude", "elevation"] if c in df.columns]
    meta = df[meta_cols].drop_duplicates("station_id").dropna(subset=["latitude", "longitude"])
    plt.figure(figsize=(8, 6))
    if "split" in meta.columns:
        for split_name, g in meta.groupby("split"):
            plt.scatter(g["longitude"], g["latitude"], s=55, label=split_name)
            for _, r in g.iterrows():
                plt.annotate(str(r["station_id"]), (r["longitude"], r["latitude"]),
                             xytext=(4, 3), textcoords="offset points", fontsize=8)
        plt.legend()
    else:
        plt.scatter(meta["longitude"], meta["latitude"], s=55)
    plt.xlabel("longitude")
    plt.ylabel("latitude")
    plt.title("Selected weather stations (geographic coordinates)")
    plt.grid(alpha=.2)
    plt.tight_layout()
    plt.show()


def plot_station_coverage(df):
    agg = {"start": ("date", "min"), "end": ("date", "max"), "n_days": ("date", "size")}
    coverage = df.groupby("station_id").agg(**agg).reset_index()
    meta_cols = [c for c in ["station_id", "station_name", "region_group", "split"] if c in df.columns]
    meta = df[meta_cols].drop_duplicates("station_id")
    coverage = coverage.merge(meta, on="station_id", how="left").sort_values("start")

    plt.figure(figsize=(11, max(4, .42 * len(coverage))))
    for y, (_, r) in enumerate(coverage.iterrows()):
        plt.hlines(y, r["start"], r["end"], linewidth=5)
    labels = [
        (f"{r.get('station_name', '')} [{r['station_id']}]" if str(r.get('station_name', '')).strip()
         else str(r["station_id"]))
        for _, r in coverage.iterrows()
    ]
    plt.yticks(range(len(coverage)), labels)
    plt.xlabel("date")
    plt.title("Observed temporal coverage by station")
    plt.grid(axis="x", alpha=.2)
    plt.tight_layout()
    plt.show()
    return coverage


def choose_representative_stations(df, max_stations=4):
    meta_cols = [c for c in ["station_id", "station_name", "region_group", "split"] if c in df.columns]
    meta = df[meta_cols].drop_duplicates("station_id")
    selected = []
    if "split" in meta.columns:
        for _, g in meta.groupby("split", sort=False):
            selected.append(str(g.iloc[0]["station_id"]))
    for sid in meta["station_id"].astype(str):
        if sid not in selected:
            selected.append(sid)
        if len(selected) >= max_stations:
            break
    return selected[:max_stations]


def plot_target_series(df, station_ids=None, start=None, end=None):
    if station_ids is None:
        station_ids = choose_representative_stations(df)
    view = df[df["station_id"].astype(str).isin([str(s) for s in station_ids])].copy()
    if start is not None:
        view = view[view["date"] >= pd.Timestamp(start)]
    if end is not None:
        view = view[view["date"] <= pd.Timestamp(end)]

    for target, ylabel, title in [
        ("temp", "°C", "Daily mean temperature"),
        ("prcp", "mm/day", "Daily precipitation"),
        ("wspd", "km/h", "Daily mean wind speed"),
    ]:
        plt.figure(figsize=(12, 4))
        for sid, g in view.groupby("station_id"):
            name = sid
            if "station_name" in g.columns and g["station_name"].notna().any():
                name = f"{g['station_name'].dropna().iloc[0]} ({sid})"
            plt.plot(g["date"], g[target], linewidth=1, label=name)
        plt.xlabel("date")
        plt.ylabel(ylabel)
        plt.title(title)
        plt.legend(ncol=2, fontsize=8)
        plt.grid(alpha=.2)
        plt.tight_layout()
        plt.show()


def plot_monthly_climatology(df):
    work = df.copy()
    work["month"] = work["date"].dt.month
    group_cols = ["month"] + (["region_group"] if "region_group" in work.columns else [])
    clim = work.groupby(group_cols)[["temp", "prcp", "wspd"]].mean().reset_index()

    for target, ylabel, title in [
        ("temp", "mean °C", "Monthly temperature climatology"),
        ("prcp", "mean mm/day", "Monthly precipitation climatology"),
        ("wspd", "mean km/h", "Monthly wind-speed climatology"),
    ]:
        plt.figure(figsize=(8, 3.5))
        if "region_group" in clim.columns:
            for region, g in clim.groupby("region_group"):
                plt.plot(g["month"], g[target], marker="o", label=region)
            plt.legend()
        else:
            plt.plot(clim["month"], clim[target], marker="o")
        plt.xticks(range(1, 13))
        plt.xlabel("month")
        plt.ylabel(ylabel)
        plt.title(title)
        plt.grid(alpha=.2)
        plt.tight_layout()
        plt.show()


def plot_missingness(df, variables=None):
    if variables is None:
        variables = [c for c in ["temp", "tmin", "tmax", "rhum", "prcp", "wspd", "wpgt", "pres", "tsun", "cldc"] if c in df.columns]
    miss = (df.groupby("station_id")[variables]
              .apply(lambda g: g.isna().mean() * 100)
              .reset_index())
    long = miss.melt(id_vars="station_id", var_name="variable", value_name="missing_pct")
    pivot = long.pivot(index="station_id", columns="variable", values="missing_pct")
    plt.figure(figsize=(max(8, .8 * len(pivot.columns)), max(4, .42 * len(pivot))))
    im = plt.imshow(pivot.to_numpy(), aspect="auto", interpolation="nearest")
    plt.colorbar(im, label="missing values (%)")
    plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=45, ha="right")
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.title("Missing-data percentage by station and variable")
    plt.tight_layout()
    plt.show()
    return pivot


def run_weather_eda(df):
    summarize_weather_data(df)
    plot_station_locations(df)
    coverage = plot_station_coverage(df)
    # Recent years keep the multi-station time-series plots readable.
    recent_start = max(df["date"].min(), df["date"].max() - pd.DateOffset(years=2))
    plot_target_series(df, start=recent_start)
    plot_monthly_climatology(df)
    missing = plot_missingness(df)
    return coverage, missing


# Run after H1 has created df_raw (development data only).
if "df_raw" in globals():
    weather_coverage, weather_missingness = run_weather_eda(df_raw)
else:
    print("Load the supplied/downloaded weather data in H1, then rerun this cell.")


### H2A.1 Explicit verification of the raw weather dataset **[PROVIDED]**

The next cell is a **sanity check**, not an exercise. It verifies that the downloaded/provided file really contains the expected stations, date ranges and train/test roles before any interpolation, scaling or window construction. It also visualizes the temporal boundaries used later in the homework.

Check in particular that:

- each `(station_id, date)` pair is unique;
- training stations have historical data before `TRAIN_END`;
- the historical test period is `TEST_START ... TEST_END`;
- unseen Lombardy and Capitanata stations are present;
- `temp`, `prcp` and `wspd` contain plausible values and acceptable missingness.


In [ ]:
# [PROVIDED] Explicit raw-data verification before preprocessing.
def verify_raw_weather_data(df):
    work = df.copy()
    work["date"] = pd.to_datetime(work["date"]).dt.normalize()

    print("DATASET SHAPE:", work.shape)
    print("DATE RANGE:", work["date"].min().date(), "->", work["date"].max().date())
    print("N STATIONS:", work["station_id"].astype(str).nunique())
    print("TRAIN END:", TRAIN_END.date())
    print("TEST PERIOD:", TEST_START.date(), "->", TEST_END.date())
    print()

    duplicate_pairs = work.duplicated(["station_id", "date"]).sum()
    print("Duplicate (station_id, date) rows:", int(duplicate_pairs))
    assert duplicate_pairs == 0, "Duplicate station/date rows found."

    required = {"date", "station_id", "split", "temp", "prcp", "wspd"}
    missing_cols = sorted(required - set(work.columns))
    assert not missing_cols, f"Missing required columns: {missing_cols}"

    split_table = (
        work[["station_id", "split"]]
        .drop_duplicates()
        .groupby("split")["station_id"]
        .nunique()
        .rename("n_stations")
        .to_frame()
    )
    display(split_table)

    coverage = (
        work.groupby(["station_id", "split"], as_index=False)
        .agg(
            first_date=("date", "min"),
            last_date=("date", "max"),
            rows=("date", "size"),
            temp_missing=("temp", lambda s: s.isna().mean()),
            prcp_missing=("prcp", lambda s: s.isna().mean()),
            wspd_missing=("wspd", lambda s: s.isna().mean()),
        )
    )
    for c in ["temp_missing", "prcp_missing", "wspd_missing"]:
        coverage[c] = (100 * coverage[c]).round(1)
    display(coverage.sort_values(["split", "station_id"]))

    target_summary = work[["temp", "prcp", "wspd"]].describe().T
    display(target_summary)
    return coverage


def _pick_station_for_verification(df):
    meta = df[["station_id", "split"]].drop_duplicates()
    train = meta[meta["split"].astype(str).eq("train")]
    if len(train):
        return str(train.iloc[0]["station_id"])
    return str(meta.iloc[0]["station_id"])


def plot_raw_train_test_boundaries(df, station_id=None):
    work = df.copy()
    work["date"] = pd.to_datetime(work["date"]).dt.normalize()
    work["station_id"] = work["station_id"].astype(str)
    if station_id is None:
        station_id = _pick_station_for_verification(work)
    g = work[work["station_id"].eq(str(station_id))].sort_values("date")
    if g.empty:
        raise ValueError(f"Station {station_id} not found")

    title_name = str(station_id)
    if "station_name" in g.columns and g["station_name"].notna().any():
        title_name = f"{g['station_name'].dropna().iloc[0]} ({station_id})"

    for target, ylabel in [("temp", "Temperature (°C)"),
                            ("prcp", "Precipitation (mm/day)"),
                            ("wspd", "Wind speed (km/h)")]:
        plt.figure(figsize=(13, 3.3))
        plt.plot(g["date"], g[target], linewidth=.8)
        plt.axvline(TRAIN_END, linestyle="--", linewidth=1.5, label="train/dev end")
        plt.axvline(TEST_START, linestyle=":", linewidth=1.5, label="historical test start")
        plt.axvline(TEST_END, linestyle="--", linewidth=1.5, label="historical test end")
        plt.xlabel("date")
        plt.ylabel(ylabel)
        plt.title(f"Raw {target}: {title_name} — verify train/test boundaries")
        plt.grid(alpha=.2)
        plt.legend()
        plt.tight_layout()
        plt.show()


def plot_target_ranges_by_split(df):
    work = df.copy()
    targets = [("temp", "Temperature (°C)"),
               ("prcp", "Precipitation (mm/day)"),
               ("wspd", "Wind speed (km/h)")]
    order = [x for x in ["train", "spatial_test_lombardia", "spatial_test_capitanata"]
             if x in set(work["split"].astype(str))]

    for target, ylabel in targets:
        groups = [work.loc[work["split"].astype(str).eq(s), target].dropna().to_numpy() for s in order]
        groups = [g for g in groups if len(g)]
        labels = [s for s in order if work.loc[work["split"].astype(str).eq(s), target].notna().any()]
        if not groups:
            continue
        plt.figure(figsize=(10, 3.8))
        plt.boxplot(groups, tick_labels=labels, showfliers=False)
        plt.ylabel(ylabel)
        plt.title(f"{target}: raw distribution by geographic split")
        plt.xticks(rotation=15, ha="right")
        plt.grid(axis="y", alpha=.2)
        plt.tight_layout()
        plt.show()


if "df_raw" in globals():
    raw_verification_table = verify_raw_weather_data(df_raw)
    plot_raw_train_test_boundaries(df_raw)
    plot_target_ranges_by_split(df_raw)
else:
    print("Run H1 first so that df_raw exists, then rerun this cell.")


### Interpretation prompts (no plotting code to write)

Use the provided figures to answer briefly:

1. Which variables show the strongest seasonal structure?
2. Are precipitation and wind visually more irregular than temperature?
3. Do Capitanata stations appear distributionally different from Lombardy stations?
4. Are there stations/variables with substantially poorer coverage?
5. Which visual patterns make a 60-day context plausible, and which dependencies might require longer seasonal information?


## H2B. Preprocessing **[PROVIDED]**

H2 prepares the **daily station table**. It does **not** create sequence windows or TimeDistributed labels.

The steps are:

1. regularize each station on a daily calendar;
2. add cyclic day-of-year features;
3. assign temporal/geographic phases;
4. fit imputation/scaling statistics using **training rows only**;
5. transform the complete frame using those training statistics.

We use latitude/longitude/elevation instead of a station-ID embedding so that the model can be evaluated on unseen stations. Precipitation is transformed with `log1p` for the regression target.

The historical protocol is:

- model-development data end on **2026-02-28**;
- the last `VAL_DAYS` within that period are validation data;
- historical test runs from **2026-03-01 to 2026-10-18**;
- Lombardy and Capitanata unseen stations keep separate spatial-test labels.

H3 will then convert this prepared table into `X` and `Y` sequence tensors.


In [ ]:
DYNAMIC_FEATURES = ["temp", "tmin", "tmax", "rhum", "prcp", "wspd", "wpgt", "pres", "tsun", "cldc"]
STATIC_FEATURES = ["latitude", "longitude", "elevation"]
CALENDAR_FEATURES = ["doy_sin", "doy_cos"]


def regularize_daily_grid(df):
    frames = []
    for sid, g in df.groupby("station_id", sort=False):
        g = g.sort_values("date").set_index("date")
        idx = pd.date_range(g.index.min(), g.index.max(), freq="D")
        g = g.reindex(idx); g.index.name = "date"
        g["station_id"] = sid
        for c in ["anchor", "split", *STATIC_FEATURES]:
            if c in g.columns:
                g[c] = g[c].ffill().bfill()
        for c in DYNAMIC_FEATURES:
            if c in g.columns:
                g[c] = g[c].ffill(limit=3)
        frames.append(g.reset_index())
    return pd.concat(frames, ignore_index=True)


def add_calendar_features(df):
    out = df.copy()
    doy = out["date"].dt.dayofyear.astype(float)
    out["doy_sin"] = np.sin(2*np.pi*doy/365.25)
    out["doy_cos"] = np.cos(2*np.pi*doy/365.25)
    return out


def prepare_frame(df):
    out = add_calendar_features(regularize_daily_grid(df))
    out["prcp"] = out["prcp"].clip(lower=0)
    return out

In [ ]:
VAL_START = TRAIN_END - pd.Timedelta(days=VAL_DAYS - 1)


def assign_phase(df):
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"]).dt.normalize()
    out["phase"] = "context"

    is_train_station = out["split"].astype(str).eq("train")

    # Development period on stations seen during training.
    out.loc[
        is_train_station & out["date"].lt(VAL_START),
        "phase"
    ] = "train"
    out.loc[
        is_train_station & out["date"].between(VAL_START, TRAIN_END),
        "phase"
    ] = "val"

    # Historical temporal test on the same stations.
    out.loc[
        is_train_station & out["date"].between(TEST_START, TEST_END),
        "phase"
    ] = "temporal_test"

    # Historical spatial/domain-shift tests on unseen stations.
    for split_name in ["spatial_test_lombardia", "spatial_test_capitanata"]:
        mask = (
            out["split"].astype(str).eq(split_name)
            & out["date"].between(TEST_START, TEST_END)
        )
        out.loc[mask, "phase"] = split_name

    # Anything after the historical test is reserved for later real-world evaluation.
    out.loc[out["date"].gt(TEST_END), "phase"] = "future_test"
    return out


class WeatherPreprocessor:
    def __init__(self, dynamic_features=DYNAMIC_FEATURES, static_features=STATIC_FEATURES):
        self.dynamic_features = list(dynamic_features)
        self.static_features = list(static_features)
        self.dynamic_scaler = StandardScaler()
        self.static_scaler = StandardScaler()
        self.target_scaler = StandardScaler()
        self.medians = None

    @staticmethod
    def raw_target_array(df):
        return np.c_[
            df["temp"].to_numpy(float),
            np.log1p(df["prcp"].clip(lower=0).to_numpy(float)),
            df["wspd"].to_numpy(float),
        ]

    def fit(self, train_df):
        self.dynamic_features = [c for c in self.dynamic_features if c in train_df.columns]
        self.medians = train_df[self.dynamic_features].median(numeric_only=True)
        self.dynamic_scaler.fit(train_df[self.dynamic_features].fillna(self.medians))
        self.static_scaler.fit(train_df[self.static_features].drop_duplicates())
        valid_y = train_df[["temp", "prcp", "wspd"]].dropna()
        self.target_scaler.fit(self.raw_target_array(valid_y))
        return self

    def transform_frame(self, df):
        out = df.copy()
        dyn = out[self.dynamic_features].fillna(self.medians)
        out[[f"z_{c}" for c in self.dynamic_features]] = self.dynamic_scaler.transform(dyn)
        out[[f"z_{c}" for c in self.static_features]] = self.static_scaler.transform(
            out[self.static_features]
        )
        return out

    def transform_targets(self, df):
        return self.target_scaler.transform(self.raw_target_array(df))

    def inverse_targets(self, arr):
        shape = np.asarray(arr).shape
        flat = np.asarray(arr).reshape(-1, 3)
        raw = self.target_scaler.inverse_transform(flat)
        raw[:, 1] = np.expm1(raw[:, 1]).clip(min=0)
        return raw.reshape(shape)


# Execute preprocessing once H1 has produced df_raw:
# df = assign_phase(prepare_frame(df_raw))
# prep = WeatherPreprocessor().fit(df[df.phase == "train"])
# df_z = prep.transform_frame(df)
# display(df_z[["date", "station_id", "split", "phase"]].head())


## H3. Building `WeatherTimeDistributedDataset` **[PROVIDED]**

H3 is intentionally different from H2. H2 produced a **prepared table**; H3 turns that table into PyTorch sequence samples.

For one station and a window beginning at row `i`:

```text
input rows       : i ... i+59
X shape          : [60, F]

all target rows  : i+1 ... i+60+21-1
                  (80 rows = 60 + 21 - 1)

sliding targets  :
 origin s=0      -> days i+1  ... i+21
 origin s=1      -> days i+2  ... i+22
 ...
 origin s=59     -> days i+60 ... i+80

Y shape          : [60, 21, 3]
                   60 forecast origins
                   x 21 future days
                   x 3 targets (temp, prcp, wspd)
```

This is the homework analogue of Géron's sequence-to-sequence / TimeDistributed construction. During **training**, loss is computed over all 60 origins. During realistic validation/test reporting, we use only the final origin `[:, -1]`, corresponding to the forecast made after observing all 60 input days.


In [ ]:
class WeatherTimeDistributedDataset(Dataset):
    """Build station-wise TimeDistributed samples.

    Returns
    -------
    x : FloatTensor [T, F]
        T=input_days observed feature vectors.
    y : FloatTensor [T, H, 3]
        For every input timestep s, the next H targets.
    meta : dict
        Dates and station information useful for inspection/evaluation.
    """

    def __init__(self, df, preprocessor, phase, input_days=60, horizon=21,
                 supervision_scope=None):
        self.df = df.copy()
        self.prep = preprocessor
        self.phase = phase
        self.input_days = int(input_days)
        self.horizon = int(horizon)

        # For train/validation all TimeDistributed labels must belong to the phase.
        # For test sets only the FINAL 21-day forecast must belong to the test phase,
        # allowing the 60-day context to start before TEST_START.
        if supervision_scope is None:
            supervision_scope = "all_origins" if phase in {"train", "val"} else "final_origin"
        if supervision_scope not in {"all_origins", "final_origin"}:
            raise ValueError("supervision_scope must be 'all_origins' or 'final_origin'")
        self.supervision_scope = supervision_scope

        self.feature_cols = (
            [f"z_{c}" for c in preprocessor.dynamic_features]
            + [f"z_{c}" for c in preprocessor.static_features]
            + CALENDAR_FEATURES
        )

        self.groups = {
            str(sid): g.sort_values("date").reset_index(drop=True)
            for sid, g in self.df.groupby("station_id", sort=False)
        }
        self.samples = []

        for sid, g in self.groups.items():
            max_start = len(g) - self.input_days - self.horizon + 1
            for start in range(max(0, max_start)):
                end = start + self.input_days

                x_df = g.iloc[start:end]
                # Length T+H-1. Sliding H-wide windows over this array yield T labels.
                target_period = g.iloc[start + 1:end + self.horizon]

                if len(x_df) != self.input_days or len(target_period) != self.input_days + self.horizon - 1:
                    continue
                if x_df[self.feature_cols].isna().any().any():
                    continue
                if target_period[TARGETS].isna().any().any():
                    continue

                if self.supervision_scope == "all_origins":
                    phase_ok = target_period["phase"].eq(phase).all()
                else:
                    # Final origin = last observed day; its labels are rows end ... end+H-1.
                    final_target = g.iloc[end:end + self.horizon]
                    phase_ok = (
                        len(final_target) == self.horizon
                        and final_target["phase"].eq(phase).all()
                    )

                if phase_ok:
                    self.samples.append((sid, start))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sid, start = self.samples[idx]
        g = self.groups[sid]
        end = start + self.input_days

        x_df = g.iloc[start:end]
        target_period = g.iloc[start + 1:end + self.horizon]

        x = torch.tensor(
            x_df[self.feature_cols].to_numpy(dtype=np.float32),
            dtype=torch.float32,
        )

        # [T+H-1, 3] after target scaling
        target_z = self.prep.transform_targets(target_period).astype(np.float32)
        target_z = torch.tensor(target_z, dtype=torch.float32)

        # unfold(0,H,1): [T,3,H] -> permute: [T,H,3]
        y = target_z.unfold(0, self.horizon, 1).permute(0, 2, 1).contiguous()

        final_target = g.iloc[end:end + self.horizon]
        meta = {
            "station_id": sid,
            "input_start": str(x_df.iloc[0]["date"].date()),
            "input_end": str(x_df.iloc[-1]["date"].date()),
            "origin": str(x_df.iloc[-1]["date"].date()),
            "forecast_start": str(final_target.iloc[0]["date"].date()),
            "forecast_end": str(final_target.iloc[-1]["date"].date()),
        }
        return x, y, meta


def make_loaders(train_ds, val_ds, batch_size=64):
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False),
    )


# Typical construction:
# train_ds = WeatherTimeDistributedDataset(df_z, prep, "train", INPUT_DAYS, HORIZON)
# val_ds = WeatherTimeDistributedDataset(df_z, prep, "val", INPUT_DAYS, HORIZON)
# temporal_test_ds = WeatherTimeDistributedDataset(
#     df_z, prep, "temporal_test", INPUT_DAYS, HORIZON
# )
# lombardia_test_ds = WeatherTimeDistributedDataset(
#     df_z, prep, "spatial_test_lombardia", INPUT_DAYS, HORIZON
# )
# capitanata_test_ds = WeatherTimeDistributedDataset(
#     df_z, prep, "spatial_test_capitanata", INPUT_DAYS, HORIZON
# )
#
# train_loader, val_loader = make_loaders(train_ds, val_ds)
# temporal_test_loader = DataLoader(temporal_test_ds, batch_size=64, shuffle=False)
# lombardia_test_loader = DataLoader(lombardia_test_ds, batch_size=64, shuffle=False)
# capitanata_test_loader = DataLoader(capitanata_test_ds, batch_size=64, shuffle=False)
# xb, yb, meta = next(iter(train_loader))
# print("X batch:", xb.shape)  # [B,60,F]
# print("Y batch:", yb.shape)  # [B,60,21,3]


### Inspect one TimeDistributed sample **[PROVIDED]**

This cell makes the alignment explicit. Check that the first forecast origin predicts the 21 days after the first input day, while the final forecast origin predicts the 21 days after the 60th input day.


In [ ]:
def inspect_weather_sample(dataset, idx=0):
    x, y, meta = dataset[idx]
    print("station:", meta["station_id"])
    print("input:", meta["input_start"], "->", meta["input_end"])
    print("final 21-day forecast:", meta["forecast_start"], "->", meta["forecast_end"])
    print("X shape:", tuple(x.shape))
    print("Y shape:", tuple(y.shape))
    print("first origin target shape:", tuple(y[0].shape))
    print("final origin target shape:", tuple(y[-1].shape))

    assert x.shape[0] == dataset.input_days
    assert y.shape == (dataset.input_days, dataset.horizon, N_TARGETS)

# Example:
# inspect_weather_sample(train_ds, 0)


### H3.1 Visual verification of one `60 → 21` sample **[PROVIDED]**

Before training any model, visualize one dataset sample in **original physical units**. The vertical boundary marks the final observed day. The 21 points to the right are exactly the labels used for the **final TimeDistributed origin**.

This plot is intended to catch off-by-one errors immediately: the forecast must begin on the day *after* the last input day.


In [ ]:
# [PROVIDED] Visual sanity check for WeatherTimeDistributedDataset alignment.
def plot_weather_dataset_sample(dataset, idx=0):
    x, y, meta = dataset[idx]
    sid, start = dataset.samples[idx]
    g = dataset.groups[sid]
    end = start + dataset.input_days

    context = g.iloc[start:end].copy()
    forecast_dates = g.iloc[end:end + dataset.horizon]["date"].to_numpy()
    y_final_raw = dataset.prep.inverse_targets(y[-1].numpy())  # [21,3]

    print("station:", sid)
    print("input:", context.iloc[0]["date"].date(), "->", context.iloc[-1]["date"].date())
    print("forecast:", pd.Timestamp(forecast_dates[0]).date(), "->", pd.Timestamp(forecast_dates[-1]).date())
    print("X shape:", tuple(x.shape), "Y shape:", tuple(y.shape))
    assert pd.Timestamp(forecast_dates[0]) == pd.Timestamp(context.iloc[-1]["date"]) + pd.Timedelta(days=1)

    specs = [
        ("temp", 0, "Temperature (°C)"),
        ("prcp", 1, "Precipitation (mm/day)"),
        ("wspd", 2, "Wind speed (km/h)"),
    ]
    boundary = pd.Timestamp(context.iloc[-1]["date"])

    for raw_col, target_idx, ylabel in specs:
        plt.figure(figsize=(12, 3.3))
        plt.plot(context["date"], context[raw_col], label="60-day input", linewidth=1.2)
        plt.plot(forecast_dates, y_final_raw[:, target_idx], marker="o", markersize=3,
                 label="21-day final-origin label")
        plt.axvline(boundary, linestyle="--", linewidth=1.5, label="forecast origin")
        plt.xlabel("date")
        plt.ylabel(ylabel)
        plt.title(f"TimeDistributed sample verification — {raw_col} — station {sid}")
        plt.grid(alpha=.2)
        plt.legend()
        plt.tight_layout()
        plt.show()


# Run after train_ds / val_ds / test datasets have been created.
if "train_ds" in globals() and len(train_ds) > 0:
    plot_weather_dataset_sample(train_ds, 0)
else:
    print("Create train_ds first, then rerun this cell to inspect a 60→21 sample.")


### Why the labels do not leak future inputs

At early timestep `s`, some of its 21 target days may appear later in the 60-day input window. This is still causally valid **only because the recurrent network is unidirectional**: hidden state `h_s` is computed before the network processes later input timesteps. Never replace the model with a bidirectional RNN for this training setup.

For evaluation we do not average all 60 origins; we evaluate the realistic forecast origin only:

```python
pred_final = pred[:, -1]  # [B,21,3]
y_final = y[:, -1]        # [B,21,3]
```


## H4. Baseline **[PROVIDED]**

A useful weather baseline repeats the last observed week over the 21-day horizon. Compare neural models against this rather than reporting neural error in isolation.


In [ ]:
def weekly_seasonal_baseline(raw_context_targets, horizon=21):
    x = np.asarray(raw_context_targets)  # [B,T,3]
    last_week = x[:, -7:, :]
    return np.tile(last_week, (1, math.ceil(horizon / 7), 1))[:, :horizon]

## H5. Model 1 — LSTM + TimeDistributed output

`nn.LSTM(..., batch_first=True)` returns `sequence_output` with shape `[B,T,H]`.

The output layer is:

```python
nn.Linear(hidden_size, HORIZON * N_TARGETS)
```

and is applied directly to the **3-D** sequence output. This produces `[B,T,63]`, then we reshape to `[B,T,21,3]`.

This is TimeDistributed training in PyTorch.


In [ ]:
class TimeDistributedLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, horizon=21, n_targets=3):
        super().__init__()
        self.horizon = horizon
        self.n_targets = n_targets
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True, bidirectional=False)
        self.head = nn.Linear(hidden_size, horizon * n_targets)

    def forward(self, x):
        # TODO HW 1: keep ALL recurrent outputs [B,T,H]
        seq, _ = ...
        # TODO HW 1: same Linear layer at every timestep -> [B,T,63]
        raw = ...
        B, T, _ = raw.shape
        # TODO HW 1: reshape to [B,T,21,3]
        return ...

## H6. Model 2 — causal attention + TimeDistributed output

A naive attention layer over the whole `[B,T,H]` sequence would leak future information into early TimeDistributed predictions. We therefore use **causal attention**.

For every query timestep `t`, keys with index `s > t` are masked:

\[
\alpha_{t,s}=0 \quad \text{for }s>t.
\]

The attention matrix has shape `[B,T,T]`. Row `t` contains the weights used to build the context for output timestep `t`.

We concatenate the LSTM representation and its causal context, then apply the TimeDistributed forecast head.


In [ ]:
class CausalTemporalAttention(nn.Module):
    def __init__(self, hidden_size, attention_size=32):
        super().__init__()
        self.query = nn.Linear(hidden_size, attention_size, bias=False)
        self.key = nn.Linear(hidden_size, attention_size, bias=False)
        self.value = nn.Linear(hidden_size, hidden_size, bias=False)
        self.scale = math.sqrt(attention_size)

    def forward(self, seq):
        q, k, v = self.query(seq), self.key(seq), self.value(seq)
        scores = torch.matmul(q, k.transpose(1, 2)) / self.scale
        T = seq.size(1)
        # TODO HW 2: create an upper-triangular FUTURE mask (diagonal=1)
        future_mask = ...
        # TODO HW 2: replace forbidden scores by -inf
        scores = ...
        # TODO HW 2: normalize over SOURCE TIME (last dimension)
        weights = ...
        context = torch.matmul(weights, v)
        return context, weights


class TimeDistributedLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size=64, attention_size=32,
                 horizon=21, n_targets=3):
        super().__init__()
        self.horizon, self.n_targets = horizon, n_targets
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True, bidirectional=False)
        self.attention = CausalTemporalAttention(hidden_size, attention_size)
        self.head = nn.Linear(hidden_size * 2, horizon * n_targets)

    def forward(self, x, return_attention=False):
        seq, _ = self.lstm(x)
        context, weights = self.attention(seq)
        # TODO HW 3: concatenate recurrent state and causal context
        features = ...
        raw = self.head(features)
        B, T, _ = raw.shape
        pred = raw.view(B, T, self.horizon, self.n_targets)
        return (pred, weights) if return_attention else pred

## H7. Training utilities **[PROVIDED]**

The training criterion compares **all** outputs and labels:

```text
pred, y: [B,T,21,3]
```

`SmoothL1Loss` reduces over batch, TimeDistributed origin, forecast horizon and target. Gradient clipping is useful for recurrent networks.

For model selection, validation loss is also computed TimeDistributed. For reported forecasting metrics, however, we use only the **last origin** (`[:, -1]`).


In [ ]:
def run_epoch(model, loader, optimizer=None, device=DEVICE, max_grad_norm=1.0):
    training = optimizer is not None
    model.train(training)
    criterion = nn.SmoothL1Loss()
    total, n = 0.0, 0
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        if training:
            optimizer.zero_grad(set_to_none=True)
        pred = model(x)                     # [B,T,21,3]
        loss = criterion(pred, y)           # ALL TimeDistributed origins
        if training:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
        total += loss.item() * x.size(0); n += x.size(0)
    return total / max(n, 1)


def fit_model(model, train_loader, val_loader, epochs=20, lr=1e-3,
              weight_decay=1e-4, patience=4, device=DEVICE):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_state, best_val, bad = None, float("inf"), 0
    history = []
    for epoch in range(1, epochs + 1):
        t0 = time.perf_counter()
        tr = run_epoch(model, train_loader, opt, device)
        with torch.no_grad():
            va = run_epoch(model, val_loader, None, device)
        sec = time.perf_counter() - t0
        history.append({"epoch": epoch, "train_loss": tr, "val_loss": va, "seconds": sec})
        print(f"epoch {epoch:02d} | train {tr:.4f} | val {va:.4f} | {sec:.1f}s")
        if va < best_val - 1e-5:
            best_val = va; best_state = copy.deepcopy(model.state_dict()); bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping"); break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

## H8. Operational prediction and inverse scaling **[PROVIDED]**

Training uses all `T=60` outputs; the reported forecast uses only the last one. This distinction must be explicit in the report.


In [ ]:
@torch.no_grad()
def predict_final_origin(model, loader, preprocessor, device=DEVICE):
    model.eval()
    preds, truths, station_ids, origins = [], [], [], []
    for x, y, meta in loader:
        pred_all = model(x.to(device)).cpu().numpy()  # [B,T,21,3]
        pred = pred_all[:, -1]                        # [B,21,3]
        true = y.numpy()[:, -1]                       # [B,21,3]
        preds.append(pred); truths.append(true)
        station_ids.extend(list(meta["station_id"])); origins.extend(list(meta["origin"]))
    pz = np.concatenate(preds); yz = np.concatenate(truths)
    return {
        "pred": preprocessor.inverse_targets(pz),
        "true": preprocessor.inverse_targets(yz),
        "station_id": np.asarray(station_ids),
        "origin": np.asarray(origins),
    }

## H9. Metrics by forecast horizon **[PROVIDED]**

Temperature and wind use MAE in physical units. Precipitation is evaluated both as amount (MAE) and occurrence (`prcp > 0.5 mm`) because a zero-inflated target can make amount-only metrics misleading.


In [ ]:
def horizon_metrics(result, rain_threshold=RAIN_THRESHOLD_MM):
    pred, true = result["pred"], result["true"]
    rows = []
    for h in range(pred.shape[1]):
        tp_, pp_, wp_ = pred[:,h,0], pred[:,h,1], pred[:,h,2]
        ty_, py_, wy_ = true[:,h,0], true[:,h,1], true[:,h,2]
        rain_p, rain_y = pp_ > rain_threshold, py_ > rain_threshold
        tp = np.sum(rain_p & rain_y); fp = np.sum(rain_p & ~rain_y); fn = np.sum(~rain_p & rain_y)
        precision = tp / (tp + fp + 1e-9); recall = tp / (tp + fn + 1e-9)
        f1 = 2 * precision * recall / (precision + recall + 1e-9)
        rows.append({
            "horizon_day": h+1,
            "temp_mae": np.mean(np.abs(tp_ - ty_)),
            "prcp_mae": np.mean(np.abs(pp_ - py_)),
            "wind_mae": np.mean(np.abs(wp_ - wy_)),
            "rain_f1": f1,
        })
    return pd.DataFrame(rows)


def plot_horizon_metrics(m):
    for col, ylabel in [("temp_mae","Temperature MAE °C"),
                        ("wind_mae","Wind MAE km/h"),
                        ("prcp_mae","Precipitation MAE mm/day")]:
        plt.figure(figsize=(7,3)); plt.plot(m.horizon_day, m[col], marker="o")
        plt.xlabel("forecast horizon (days)"); plt.ylabel(ylabel); plt.grid(alpha=.25)
        plt.tight_layout(); plt.show()

## H10. Required experiments

### Experiment 1 — architecture
Compare:

1. weekly seasonal baseline;
2. **TimeDistributed LSTM**;
3. **TimeDistributed LSTM + causal attention**.

Use the same train/validation split and report error as a function of horizon `+1…+21`.

### Experiment 2 — context length
Default is **60 days**. Compare it with **30 days** (or, if compute permits, 90 days), changing only the context length.

Do not conduct a broad hyperparameter search.


In [ ]:
# COMPLETE example experiment scaffold.
# train_ds = WeatherTimeDistributedDataset(df_z, prep, "train", INPUT_DAYS, HORIZON)
# val_ds = WeatherTimeDistributedDataset(df_z, prep, "val", INPUT_DAYS, HORIZON)
# train_loader, val_loader = make_loaders(train_ds, val_ds, batch_size=64)
# n_features = len(train_ds.feature_cols)
#
# td_lstm = TimeDistributedLSTM(n_features, hidden_size=64, horizon=HORIZON)
# td_lstm, hist_lstm = fit_model(td_lstm, train_loader, val_loader)
#
# td_attn = TimeDistributedLSTMAttention(n_features, hidden_size=64, attention_size=32,
#                                        horizon=HORIZON)
# td_attn, hist_attn = fit_model(td_attn, train_loader, val_loader)
#
# result_lstm = predict_final_origin(td_lstm, val_loader, prep)
# result_attn = predict_final_origin(td_attn, val_loader, prep)
# plot_horizon_metrics(horizon_metrics(result_lstm))
# plot_horizon_metrics(horizon_metrics(result_attn))

## H11. Unseen-station test

Use the selected frozen model on stations whose IDs never appear during fitting. Do **not** use station-ID embeddings. Break down error by station and discuss whether latitude, longitude and elevation were sufficient to transfer.


In [ ]:
def station_metrics(result):
    rows = []
    for sid in np.unique(result["station_id"]):
        m = result["station_id"] == sid
        p, y = result["pred"][m], result["true"][m]
        rows.append({
            "station_id": sid,
            "n_windows": int(m.sum()),
            "temp_mae": np.mean(np.abs(p[...,0] - y[...,0])),
            "prcp_mae": np.mean(np.abs(p[...,1] - y[...,1])),
            "wind_mae": np.mean(np.abs(p[...,2] - y[...,2])),
        })
    return pd.DataFrame(rows).sort_values("temp_mae")

## H12. Attention inspection

For a TimeDistributed attention model the weight tensor is `[B,T,T]`. For the **operational final forecast**, inspect only the last query row:

```python
weights[:, -1, :]   # how the final origin attends over its 60 observed days
```

Attention weights can describe model focus; they are **not** proof of causality or physical feature importance.


In [ ]:
def plot_final_attention(weights, title="Final-origin causal attention"):
    w = np.asarray(weights).squeeze()
    if w.ndim == 2:  # [T,T]
        w = w[-1]
    plt.figure(figsize=(10,3)); plt.plot(np.arange(1, len(w)+1), w)
    plt.xlabel("input day within 60-day context"); plt.ylabel("attention weight")
    plt.title(title); plt.tight_layout(); plt.show()

## H13. Final comparison of all forecasting models **[PROVIDED utilities]**

The final comparison must include, at minimum:

1. weekly seasonal baseline;
2. TimeDistributed LSTM;
3. TimeDistributed LSTM + causal attention.

Evaluate the **same frozen models** on four distinct historical sets:

- **temporal validation** on seen Lombardy stations;
- **historical temporal test**: 1 March–18 October 2026 on the same stations used for training;
- **spatial test — Lombardia**: all valid test windows from Lombardy stations never seen during training;
- **spatial/domain-shift test — Capitanata**: all valid test windows from unseen Capitanata stations.

The real future test is appended later, when its ground truth becomes available.

For each historical test, use **all valid 60-day contexts whose final 21-day forecast belongs to that test period**. Do not report only one aggregate number. Show:

- a compact summary table;
- error versus forecast horizon (`+1 … +21`);
- per-station metrics for both spatial tests;
- the performance gap between the temporal test and the two spatial tests.


In [ ]:
# [PROVIDED] Inspect exactly what each evaluation dataset contains.
def summarize_evaluation_dataset(dataset, name):
    rows = []
    for sid, start in dataset.samples:
        g = dataset.groups[sid]
        end = start + dataset.input_days
        final_target = g.iloc[end:end + dataset.horizon]
        if len(final_target) != dataset.horizon:
            continue
        rows.append({
            "station_id": str(sid),
            "origin": pd.Timestamp(g.iloc[end - 1]["date"]),
            "forecast_start": pd.Timestamp(final_target.iloc[0]["date"]),
            "forecast_end": pd.Timestamp(final_target.iloc[-1]["date"]),
        })

    info = pd.DataFrame(rows)
    if info.empty:
        return pd.DataFrame([{
            "dataset": name,
            "n_windows": 0,
            "n_stations": 0,
            "first_forecast": pd.NaT,
            "last_forecast": pd.NaT,
        }])

    return pd.DataFrame([{
        "dataset": name,
        "n_windows": len(info),
        "n_stations": info["station_id"].nunique(),
        "first_forecast": info["forecast_start"].min(),
        "last_forecast": info["forecast_end"].max(),
    }])


# Once H3 datasets have been created, run this BEFORE model evaluation:
# evaluation_coverage = pd.concat([
#     summarize_evaluation_dataset(val_ds, "validation"),
#     summarize_evaluation_dataset(temporal_test_ds, "temporal test Mar-Oct 2026"),
#     summarize_evaluation_dataset(lombardia_test_ds, "unseen Lombardy"),
#     summarize_evaluation_dataset(capitanata_test_ds, "unseen Capitanata"),
# ], ignore_index=True)
# display(evaluation_coverage)


In [ ]:
# [PROVIDED] Helpers for a consistent final comparison.
def weekly_baseline_result_from_dataset(dataset):
    preds, truths, station_ids, origins = [], [], [], []

    for sid, start in dataset.samples:
        g = dataset.groups[sid]
        end = start + dataset.input_days
        x_df = g.iloc[start:end]
        y_df = g.iloc[end:end + dataset.horizon]

        context = x_df[TARGETS].copy()
        context = context.ffill()
        for c in TARGETS:
            if context[c].isna().any():
                fallback = dataset.prep.medians.get(c, 0.0) if dataset.prep.medians is not None else 0.0
                context[c] = context[c].fillna(fallback)

        pred = weekly_seasonal_baseline(
            context[TARGETS].to_numpy(float)[None, ...],
            horizon=dataset.horizon,
        )[0]
        true = y_df[TARGETS].to_numpy(float)

        preds.append(pred)
        truths.append(true)
        station_ids.append(str(sid))
        origins.append(str(x_df.iloc[-1]["date"].date()))

    return {
        "pred": np.asarray(preds),
        "true": np.asarray(truths),
        "station_id": np.asarray(station_ids),
        "origin": np.asarray(origins),
    }


def summarize_forecast_result(result):
    m = horizon_metrics(result)
    return {
        "temp_mae": m["temp_mae"].mean(),
        "prcp_mae": m["prcp_mae"].mean(),
        "wind_mae": m["wind_mae"].mean(),
        "rain_f1": m["rain_f1"].mean(),
    }


def build_model_comparison(results_by_split):
    rows = []
    for split_name, model_results in results_by_split.items():
        for model_name, result in model_results.items():
            row = {"split": split_name, "model": model_name}
            row.update(summarize_forecast_result(result))
            rows.append(row)
    return pd.DataFrame(rows)


def plot_model_horizon_comparison(model_results, metric="temp_mae", title=None):
    plt.figure(figsize=(8, 4))
    for model_name, result in model_results.items():
        m = horizon_metrics(result)
        plt.plot(m["horizon_day"], m[metric], marker="o", label=model_name)
    ylabel = {
        "temp_mae": "temperature MAE (°C)",
        "prcp_mae": "precipitation MAE (mm/day)",
        "wind_mae": "wind MAE (km/h)",
        "rain_f1": "rain occurrence F1",
    }.get(metric, metric)
    plt.xlabel("forecast horizon (days)")
    plt.ylabel(ylabel)
    plt.title(title or f"Model comparison — {metric}")
    plt.legend()
    plt.grid(alpha=.25)
    plt.tight_layout()
    plt.show()


def plot_summary_model_comparison(comparison_df, metric="temp_mae"):
    pivot = comparison_df.pivot(index="model", columns="split", values=metric)
    ax = pivot.plot(kind="bar", figsize=(10, 4))
    ax.set_ylabel(metric)
    ax.set_title(f"Overall model comparison — {metric}")
    ax.grid(axis="y", alpha=.25)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def spatial_gap_table(comparison_df, reference_split="temporal test Mar-Oct 2026"):
    """Positive MAE gaps mean degradation relative to the temporal test.
    Positive rain_f1_gap means improvement; negative means degradation.
    """
    metrics = ["temp_mae", "prcp_mae", "wind_mae", "rain_f1"]
    rows = []
    ref = comparison_df[comparison_df["split"].eq(reference_split)].set_index("model")
    for spatial_split in ["unseen Lombardy", "unseen Capitanata"]:
        cur = comparison_df[comparison_df["split"].eq(spatial_split)].set_index("model")
        common = ref.index.intersection(cur.index)
        for model in common:
            row = {"model": model, "comparison": spatial_split}
            for metric in metrics:
                row[f"{metric}_gap"] = cur.loc[model, metric] - ref.loc[model, metric]
            rows.append(row)
    return pd.DataFrame(rows)


# Build every historical evaluation loader. Test loaders never shuffle.
# temporal_test_loader = DataLoader(temporal_test_ds, batch_size=64, shuffle=False)
# lombardia_test_loader = DataLoader(lombardia_test_ds, batch_size=64, shuffle=False)
# capitanata_test_loader = DataLoader(capitanata_test_ds, batch_size=64, shuffle=False)
#
# results_by_split = {
#     "validation": {
#         "Weekly baseline": weekly_baseline_result_from_dataset(val_ds),
#         "TD LSTM": predict_final_origin(td_lstm, val_loader, prep),
#         "TD LSTM + attention": predict_final_origin(td_attn, val_loader, prep),
#     },
#     "temporal test Mar-Oct 2026": {
#         "Weekly baseline": weekly_baseline_result_from_dataset(temporal_test_ds),
#         "TD LSTM": predict_final_origin(td_lstm, temporal_test_loader, prep),
#         "TD LSTM + attention": predict_final_origin(td_attn, temporal_test_loader, prep),
#     },
#     "unseen Lombardy": {
#         "Weekly baseline": weekly_baseline_result_from_dataset(lombardia_test_ds),
#         "TD LSTM": predict_final_origin(td_lstm, lombardia_test_loader, prep),
#         "TD LSTM + attention": predict_final_origin(td_attn, lombardia_test_loader, prep),
#     },
#     "unseen Capitanata": {
#         "Weekly baseline": weekly_baseline_result_from_dataset(capitanata_test_ds),
#         "TD LSTM": predict_final_origin(td_lstm, capitanata_test_loader, prep),
#         "TD LSTM + attention": predict_final_origin(td_attn, capitanata_test_loader, prep),
#     },
# }
#
# final_comparison = build_model_comparison(results_by_split)
# display(final_comparison.sort_values(["split", "temp_mae"]))
#
# # Complete spatial evaluation: inspect every unseen station separately.
# for split_name in ["unseen Lombardy", "unseen Capitanata"]:
#     print(f"\nPer-station results — {split_name}")
#     for model_name, result in results_by_split[split_name].items():
#         print(f"\n{model_name}")
#         display(station_metrics(result))
#
# # Error/F1 versus forecast horizon for every evaluation split.
# for split_name, model_results in results_by_split.items():
#     for metric in ["temp_mae", "wind_mae", "prcp_mae", "rain_f1"]:
#         plot_model_horizon_comparison(
#             model_results, metric,
#             title=f"{split_name} — {metric} by forecast horizon",
#         )
#
# # Aggregate comparison across validation + all historical tests.
# for metric in ["temp_mae", "wind_mae", "prcp_mae", "rain_f1"]:
#     plot_summary_model_comparison(final_comparison, metric)
#
# # Quantify the extra cost of spatial transfer and domain shift.
# display(spatial_gap_table(final_comparison))


### What to discuss from the final comparison

Your discussion should distinguish at least five questions:

- **Temporal generalization:** how much does performance change from validation to the March–October 2026 temporal test on seen stations?
- **Architecture:** does attention improve the TimeDistributed LSTM, and for which target?
- **Horizon:** which models degrade fastest between day `+1` and day `+21`?
- **Spatial transfer:** how much performance is lost on unseen Lombardy stations, both overall and station by station?
- **Domain shift:** is the degradation larger on Capitanata than on unseen Lombardy stations?

Use the spatial-gap table to quantify these effects rather than discussing them only qualitatively. When the future ground truth is released, append it as another split rather than replacing any historical test.


## H14. OPTIONAL — stationarity analysis

This section is **not required** for the homework. Students who want to connect the weather data to the live discussion may repeat rolling-statistics, ACF, ADF and KPSS for one station/target.

Suggested questions:

- How strongly does annual seasonality affect temperature stationarity tests?
- Is wind qualitatively different?
- Why are ADF/KPSS harder to interpret for zero-inflated precipitation?
- Would removing a climatological seasonal component change the forecasting task?


In [ ]:
# OPTIONAL example (requires statsmodels imports from the LIVE section):
# sid = station_meta.query("split == 'train'").station_id.iloc[0]
# temp_series = df_raw.query("station_id == @sid").set_index("date")["temp"]
# stationarity_report(temp_series.dropna(), rolling=30, acf_lags=90,
#                     title=f"Temperature station {sid}")

## H15. Submission and real future test

Submit:

- notebook with outputs and discussion;
- frozen model weights;
- preprocessing/configuration needed for inference;
- a function that accepts the latest 60-day context and returns `[21,3]` per station.

After submission, the instructor runs the **frozen** model on the latest observation context and records forecasts for the next 21 days. When those observations become available, an official ground-truth CSV is frozen and scored.

The future test should have modest grading weight: a history-only recurrent model at a 21-day horizon mostly learns persistence/climatology and is not comparable to numerical weather prediction.


In [ ]:
def build_future_forecast_table(station_ids, origin_date, predictions):
    """predictions: [n_stations,21,3] in physical units."""
    rows = []
    origin = pd.Timestamp(origin_date)
    for i, sid in enumerate(station_ids):
        for h in range(HORIZON):
            rows.append({
                "station_id": str(sid),
                "forecast_origin": origin.date().isoformat(),
                "target_date": (origin + pd.Timedelta(days=h+1)).date().isoformat(),
                "horizon_day": h+1,
                "temp": float(predictions[i,h,0]),
                "prcp": max(0.0, float(predictions[i,h,1])),
                "wspd": max(0.0, float(predictions[i,h,2])),
            })
    return pd.DataFrame(rows)

## Required final discussion

Answer concisely but substantively:

1. Does TimeDistributed LSTM beat the weekly baseline?
2. Does causal attention improve every target equally?
3. How does error change from day `+1` to `+21`?
4. Which target is hardest and why?
5. How different are validation and the **March–October 2026 temporal test** on the same stations?
6. Does the selected model generalize to unseen Lombardy stations? Which unseen station is hardest?
7. Is the domain shift to Capitanata larger than the spatial shift within Lombardia? Quantify the difference.
8. What changed when context was reduced from 60 to 30 days (or increased to 90)?
9. Why does TimeDistributed training provide more supervision than loss only at the final timestep?
10. Why must attention be causal when the loss is computed at every input timestep?
11. Why is a 21-day history-only model not equivalent to an NWP weather forecast?
12. What did the initial weather visualizations suggest about seasonality, missingness and regional differences?
13. Which model gives the best overall trade-off across temporal test, unseen Lombardia and Capitanata, and does the ranking change with forecast horizon?


## References

- Aurélien Géron, *Hands-On Machine Learning*, Chapter 13 companion notebook: https://colab.research.google.com/github/serivan/DeepLearning/blob/master/PyTorch/13_processing_sequences_using_rnns_and_cnns.ipynb
- PyTorch documentation: `nn.RNN`, `nn.LSTM`, `nn.Linear`, tensor `unfold`.
- Meteostat documentation for daily station observations and dataset preparation.
